#Importing Libraries

In [4]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')
%matplotlib inline

import tensorflow as tf
from keras.preprocessing.image import load_img
from keras.models import Model
from tensorflow import keras
from keras.layers import *
from tensorflow.keras.utils import plot_model

ModuleNotFoundError: No module named 'tensorflow'

#Setting Dataset Path

In [ ]:
#Set data path
dataset_path = "C:\Users\This PC\Downloads\Computer Vision\UTKFace"

#Extracting Labels from Filenames

In [ ]:
# labels - age, gender, ethnicity
image_paths = []
age_labels = []
gender_labels = []

for filename in tqdm(os.listdir(dataset_path)):
    image_path = os.path.join(dataset_path, filename)
    temp = filename.split('_')
    age = int(temp[0])
    gender = int(temp[1])
    image_paths.append(image_path)
    age_labels.append(age)
    gender_labels.append(gender)


0it [00:00, ?it/s]

In [ ]:
#Convert to dataframe
df = pd.DataFrame()
df['image'], df['age'], df['gender'] = image_paths, age_labels, gender_labels
df.head()

In [ ]:
#example on how to open one image
from PIL import Image
img = Image.open(df['image'][12])
plt.axis('off')
plt.imshow(img);

In [ ]:
sns.distplot(df['age'], color = 'red')

In [ ]:
gender_dict ={0: 'Male', 1: 'Female'}

In [ ]:
#to display grid of images
plt.figure(figsize=(10,10))
files = df.iloc[:10]

for index, file , age, gender in files.itertuples():
    plt.subplot(5,2,index+1)
    img = load_img(file)
    img = np.array(img)
    plt.imshow(img)
    plt.title(f"Age: {age} Gender: {gender_dict[gender]}")
    plt.axis('off')

#Extracting Image Features

In [ ]:
def extract_features(images):
  features=[]
  for image in tqdm(images):
    img = load_img(image, color_mode ="grayscale")
    img = img.resize((128,128), Image.Resampling.LANCZOS)
    img = np.array(img)
    features.append(img)

  features = np.array(features)
  features = features.reshape(len(features), 128, 128, 1)
  return features

In [ ]:
X = extract_features(df['image'])

In [ ]:
X.shape

In [ ]:
X = X/255.0

In [ ]:
y_gender = np.array(df['gender'])
y_age = np.array(df['age'])

In [ ]:
input_shape = (128, 128, 1)

#Building the CNN model

In [ ]:
from tensorflow.keras.layers import BatchNormalization


In [ ]:
inputs = Input((input_shape))
# convolutional layers
conv_1 = Conv2D(32, kernel_size=(3, 3), activation=None)(inputs)
x = BatchNormalization()(conv_1)
x = ReLU()(x)
maxp1 = MaxPooling2D(pool_size = (2,2))(x)

conv_2 = Conv2D(64, kernel_size=(3, 3), activation=None)(maxp1)
x = BatchNormalization()(conv_2)
x = ReLU()(x)
maxp2 = MaxPooling2D(pool_size = (2,2))(x)

conv_3 = Conv2D(128, kernel_size=(3, 3), activation=None)(maxp2)
x = BatchNormalization()(conv_3)
x = ReLU()(x)
maxp3 = MaxPooling2D(pool_size = (2,2))(x)

conv_4 = Conv2D(256, kernel_size=(3, 3), activation=None)(maxp3)
x = BatchNormalization()(conv_4)
x = ReLU()(x)
maxp4 = MaxPooling2D(pool_size = (2,2))(x)

flatten = Flatten()(maxp4)

#fully connected layers
dense_1 = Dense(256, activation='relu')(flatten)
dense_2 = Dense(256, activation='relu')(flatten)

dropout_1 = Dropout(0.4)(dense_1)
dropout_2 = Dropout(0.4)(dense_2)

output_1 = Dense(1, activation='sigmoid', name='gender_output')(dropout_1)
output_2 = Dense(1, activation='relu', name='age_output')(dropout_2)

model = Model(inputs=[inputs], outputs=[output_1, output_2])

model.compile(loss=['binary_crossentropy', 'mae'], optimizer='adam', metrics=['accuracy','mae'])


In [ ]:
model.summary()

In [ ]:
plot_model(
    model,
    show_shapes=True,
    dpi = 80,
    show_layer_activations = True,
    show_trainable = True
)

In [ ]:
#Train the model with separate metrics for gender and age outputs
history = model.fit(
    x = X,
    y = [y_gender, y_age],
    batch_size = 32,
    epochs = 30,
    validation_split = 0.2
)

#Accuracy Curve

In [ ]:
# plot results for gender
acc = history.history['gender_out_accuracy']
val_acc = history.history['val_gender_out_accuracy']
epochs = range(len(acc))

plt.plot(epochs, acc, 'b', label='Training accuracy')
plt.plot(epochs, val_acc, 'r', label='Validation accuracy')
plt.title('Accuracy Graph')
plt.legend()
plt.figure()

loss = history.history['loss']
val_loss = history.history['val_loss']

plt.plot(epochs, loss, 'b', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Loss Graph')
plt.legend()
plt.show()

In [ ]:
#plot results for age
loss = history.history['age_out_mae']
val_loss = history.history['val_age_out_mae']
epochs = range(len(loss))

plt.plot(epochs, loss, 'b', label='Training MAE')
plt.plot(epochs, val_loss, 'r', label='Validation MAE')
plt.title('Loss Graph')
plt.legend()
plt.show()

#Saving the model

In [ ]:
import pickle
pickle,dump(model, open('model.pkl', 'wb'))